# 11 · Graph Transformers

## What this notebook covers
Graph Transformers are the current frontier: they combine the global attention of transformers with graph structure. Standard transformers treat each token as a node in a fully-connected graph; graph transformers add structural bias (positional encodings, edge features) to make attention structure-aware.

## The motivation
Standard GNNs have two limitations:
1. **Over-smoothing**: after many message-passing layers, all node embeddings converge to the same value
2. **Bottleneck**: information must flow through paths in the graph — distant nodes can't communicate directly

Transformers solve both by allowing every node to attend directly to every other node. The challenge is encoding the *structural position* of each node, since transformers have no inherent notion of graph topology.

## Positional encodings for graphs
| Encoding | Description |
|---------|------------|
| **Laplacian PE (LPE)** | Eigenvectors of normalised Laplacian; encode global topology |
| **Random Walk PE (RWPE)** | Diagonal of k-step random walk matrix; local structure |
| **Distance encoding** | Shortest path distances between node pairs |

**GPS (General, Powerful, Scalable)** (Rampášek et al. 2022) is the current state of the art: it combines local message passing (MPNN) with global attention in each layer, using Laplacian or RWPE positional encodings. It achieves best-in-class results on LRGB (Long Range Graph Benchmark) while scaling to large graphs.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
torch.manual_seed(42); np.random.seed(42)

try:
    from torch_geometric.datasets import ZINC
    from torch_geometric.nn import GPSConv, GINEConv, global_add_pool
    from torch_geometric.loader import DataLoader
    HAS_PYG_GPS = True
except (ImportError, AttributeError):
    HAS_PYG_GPS = False

print(f"PyTorch Geometric GPS available: {HAS_PYG_GPS}")

In [ ]:
# ── Laplacian positional encoding ─────────────────────────────────────────────
def laplacian_pe(G, k=8):
    """
    Compute first k non-trivial eigenvectors of the normalised Laplacian.
    Used as structural positional encodings in graph transformers.
    """
    n = G.number_of_nodes()
    A = nx.to_numpy_array(G)
    D = np.diag(A.sum(1))
    D_inv_sqrt = np.diag(1/np.sqrt(A.sum(1) + 1e-8))
    L_norm = np.eye(n) - D_inv_sqrt @ A @ D_inv_sqrt
    eigenvalues, eigenvectors = np.linalg.eigh(L_norm)
    # Skip the trivial eigenvector (eigenvalue ≈ 0)
    pe = eigenvectors[:, 1:k+1]  # (N, k)
    return eigenvalues[1:k+1], pe

G = nx.karate_club_graph()
evals, pe = laplacian_pe(G, k=8)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pos = nx.spring_layout(G, seed=42)
for i, ev_idx in enumerate([0, 1, 2]):
    nx.draw(G, pos=pos, ax=axes[i],
            node_color=pe[:, ev_idx], cmap="RdBu", vmin=-pe[:,ev_idx].max(), vmax=pe[:,ev_idx].max(),
            with_labels=False, node_size=100, edge_color="grey", alpha=0.8)
    axes[i].set_title(f"Laplacian PE dim {ev_idx+1}  (λ={evals[ev_idx]:.3f})")
plt.suptitle("Laplacian Positional Encodings — structural node positions")
plt.tight_layout()
plt.savefig(f"{base}/11_graph_transformer_pe.png", dpi=120)
plt.show()
print(f"First 8 non-trivial eigenvalues: {evals.round(4)}")

In [ ]:
# ── Simplified Graph Transformer from scratch ─────────────────────────────────
# Node feature = identity + Laplacian PE
n = G.number_of_nodes()
X_id = np.eye(n)
X_with_pe = np.concatenate([X_id, pe], axis=1)
X_t = torch.FloatTensor(X_with_pe)
y_kc = torch.LongTensor([0 if G.nodes[i]["club"]=="Mr. Hi" else 1 for i in range(n)])

class GraphTransformerLayer(nn.Module):
    """
    Single Graph Transformer layer:
    - Global multi-head self-attention (all pairs)
    - Local MPNN aggregation (neighbours only)
    - Combine both
    """
    def __init__(self, d_model, n_heads, adj):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.local = nn.Linear(d_model, d_model)
        self.combine = nn.Linear(d_model * 2, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.adj = adj  # adjacency mask for local aggregation

    def forward(self, X):
        # Global attention (all-pairs)
        X_q = X.unsqueeze(0)  # (1, N, D)
        attn_out, _ = self.attn(X_q, X_q, X_q)
        attn_out = attn_out.squeeze(0)  # (N, D)

        # Local aggregation: mean of neighbours
        local_agg = self.adj @ X / (self.adj.sum(1, keepdim=True) + 1e-8)
        local_out = F.relu(self.local(local_agg))

        combined = self.combine(torch.cat([attn_out, local_out], dim=-1))
        return self.norm2(X + F.relu(combined))

A_norm = torch.FloatTensor(nx.to_numpy_array(G) + np.eye(n))

class SimpleGT(nn.Module):
    def __init__(self, in_d, d_model, out_d):
        super().__init__()
        self.proj = nn.Linear(in_d, d_model)
        self.layer1 = GraphTransformerLayer(d_model, 4, A_norm)
        self.layer2 = GraphTransformerLayer(d_model, 4, A_norm)
        self.head   = nn.Linear(d_model, out_d)
    def forward(self, X):
        X = F.relu(self.proj(X))
        X = self.layer1(X); X = self.layer2(X)
        return self.head(X)

gt_model = SimpleGT(X_with_pe.shape[1], 32, 2)
opt = torch.optim.Adam(gt_model.parameters(), lr=5e-3, weight_decay=1e-4)
for _ in range(300):
    out = gt_model(X_t)
    loss = F.cross_entropy(out, y_kc)
    opt.zero_grad(); loss.backward(); opt.step()

gt_model.eval()
with torch.no_grad():
    preds = gt_model(X_t).argmax(dim=1)
acc = (preds == y_kc).float().mean().item()
print(f"Graph Transformer + Laplacian PE on Karate Club: accuracy={acc:.3f}")